# Day 4 - Local PostgreSQL in Docker

**Goal:** Run PostgreSQL in a Docker container, connect from Python, and load cleaned data into a table.

This notebook is generic and works with the apprentice's own scraper output.

## 1) Start PostgreSQL in Docker

Run this in a terminal:

```bash
docker run --name postgres-apprentice \
  -e POSTGRES_USER=apprentice \
  -e POSTGRES_PASSWORD=apprentice_pw \
  -e POSTGRES_DB=apx_data \
  -p 5432:5432 \
  -d postgres:16
```

Check it with:

```bash
docker ps
```

## 2) Python packages

Make sure the environment has pandas, SQLAlchemy, and psycopg available.

In [4]:
import os
from pathlib import Path
import pandas as pd
from sqlalchemy import create_engine, text

csv_path = Path('cleaned_data.csv')
if not csv_path.exists():
    sample = [
        {'scraped_at': '2026-06-30T12:00:00', 'item': 'Example A', 'value': 10.5, 'quantity': 1, 'source_url': 'https://example.com/a'},
        {'scraped_at': '2026-06-30T12:05:00', 'item': 'Example B', 'value': 7.0, 'quantity': 2, 'source_url': 'https://example.com/b'},
    ]
    pd.DataFrame(sample).to_csv(csv_path, index=False)
    print('Wrote sample cleaned_data.csv')

df = pd.read_csv(csv_path, parse_dates=['scraped_at'])
df.head()


Wrote sample cleaned_data.csv


,scraped_at,item,value,quantity,source_url
0,2026-06-30 12:00:00,Example A,10.5,1,https://example.com/a
1,2026-06-30 12:05:00,Example B,7.0,2,https://example.com/b


In [ ]:
db_user = os.environ.get('DB_USER', 'apprentice')
db_pass = os.environ.get('DB_PASS', 'apprentice_pw')
db_host = os.environ.get('DB_HOST', 'localhost')
db_port = os.environ.get('DB_PORT', '5432')
db_name = os.environ.get('DB_NAME', 'apx_data')

conn_str = f'postgresql+psycopg://{db_user}:{db_pass}@{db_host}:{db_port}/{db_name}'
print(conn_str)

engine = create_engine(conn_str)
with engine.connect() as conn:
    result = conn.execute(text('select 1'))
    print(result.fetchone())


In [ ]:
df.to_sql('records', engine, if_exists='replace', index=False)
print('Loaded rows:', len(df))


In [ ]:
check = pd.read_sql('select count(*) as row_count from records', engine)
check


## 3) What the apprentice should do
- Start the container.
- Confirm the Python connection works.
- Load the cleaned CSV into a table.
- Run a row count query.
- Try one extra query on their own.
